In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# ── Load API key ──────────────────────────────────────────────────────────────
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
print("OpenAI key loaded:", "✅" if api_key else "❌ MISSING")

ROOT       = Path.cwd().parent
VECTOR_DIR = ROOT / 'vectorstore'

OpenAI key loaded: ✅


In [2]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# ── Must use same embedding model as Phase 2 ──────────────────────────────────
# Different model = wrong vectors = garbage retrieval results
embeddings = OpenAIEmbeddings(
    model='text-embedding-3-small',
    openai_api_key=api_key
)

vectorstore = Chroma(
    persist_directory=str(VECTOR_DIR),
    embedding_function=embeddings
)

print(f"Vector store loaded ✅")
print(f"Total vectors: {vectorstore._collection.count()}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store loaded ✅
Total vectors: 271


In [3]:
# ── MMR Retriever ─────────────────────────────────────────────────────────────
# MMR = Maximal Marginal Relevance
# Fetches 10 candidates, returns 4 most relevant AND diverse chunks.
# Prevents returning 4 chunks that all say the same thing.

retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 4, 'fetch_k': 10}
)

# ── Quick test ────────────────────────────────────────────────────────────────
test = retriever.invoke("Dubai property market")
print(f"Retriever working ✅ — returned {len(test)} chunks")
for i, doc in enumerate(test):
    print(f"  Chunk {i+1}: {Path(doc.metadata.get('source','?')).name} "
          f"p.{doc.metadata.get('page','?')} — "
          f"{doc.page_content[:80]}...")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Retriever working ✅ — returned 4 chunks
  Chunk 1: Dubai-Residential-Market-Performance-Q12025.pdf p.0 — Dubai
Residential
Market
Performance
Q1 2025...
  Chunk 2: july_market_report_2025.pdf p.12 — In summary, the medium-term outlook for Dubai 
real estate is one of tempered op...
  Chunk 3: dubai-residential-market-report---q2-2025.pdf p.2 — Dubai’s prime residential market reached unprecedented levels 
in Q2 2025, conti...
  Chunk 4: dubai-residential-market-review-special-edition-q3-02025.pdf p.2 — Dubai Residential Market Review - Q3 2025 Dubai Residential Market Review - Q3 2...


In [4]:
from langchain.prompts import PromptTemplate

# ── Prompt engineering ────────────────────────────────────────────────────────
# Three rules baked in:
#   1. ONLY use context — prevents hallucination
#   2. Cite source and page — makes answers verifiable
#   3. Say you don't know — honest fallback

PROMPT_TEMPLATE = """You are an expert UAE real estate analyst assistant.
Answer the question using ONLY the context provided below.
If the answer is not found in the context, say exactly:
"I don't have enough information in the provided documents to answer this."

Always cite which document and page number your answer comes from.

Context:
{context}

Question: {question}

Answer (with source citation):"""

prompt = PromptTemplate(
    template=PROMPT_TEMPLATE,
    input_variables=["context", "question"]
)

print("Prompt template built ✅")

Prompt template built ✅


In [5]:
from langchain_openai import ChatOpenAI

# ── GPT-3.5-turbo ─────────────────────────────────────────────────────────────
# temperature=0 → deterministic, factual answers — no randomness.
# Cost: ~$0.002 per question — negligible for a portfolio project.

llm = ChatOpenAI(
    model='gpt-3.5-turbo',
    temperature=0,
    openai_api_key=api_key
)

# ── Test connection ───────────────────────────────────────────────────────────
response = llm.invoke("Say hello in one sentence.")
print("LLM connected ✅")
print("Response:", response.content)

LLM connected ✅
Response: Hello, how are you doing today?


In [6]:
from langchain.chains import RetrievalQA

# ── RetrievalQA chain ─────────────────────────────────────────────────────────
# Combines retriever + prompt + LLM:
#   1. Question comes in
#   2. Retriever finds top 4 relevant chunks
#   3. Chunks inserted into {context} in the prompt
#   4. Full prompt sent to GPT-3.5-turbo
#   5. Answer returned with source documents

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',              # put all chunks into one prompt
    retriever=retriever,
    return_source_documents=True,    # so we can show which pages were used
    chain_type_kwargs={'prompt': prompt}
)

print("RAG chain built ✅")

RAG chain built ✅


In [7]:
def ask(question: str) -> dict:
    """Ask the RAG chain and print formatted answer + sources."""
    print(f"Q: {question}")
    print("─" * 60)

    result = qa_chain.invoke({"query": question})

    # ── Print answer ──────────────────────────────────────────────────────────
    print(f"A: {result['result']}")

    # ── Deduplicate and print sources ─────────────────────────────────────────
    # Multiple chunks may come from the same page — show each once only
    print("\n── Sources ───────────────────────────────────────────")
    seen    = set()
    sources = []
    for doc in result['source_documents']:
        source = doc.metadata.get('source', 'unknown')
        page   = doc.metadata.get('page', '?')
        key    = f"{source}::{page}"
        if key not in seen:
            seen.add(key)
            filename = Path(source).name
            sources.append({'filename': filename, 'page': page})
            print(f"  📄 {filename} — page {page}")
    print()

    return {"answer": result['result'], "sources": sources}

In [8]:
ask("What is the current state of the Dubai real estate market?")

Q: What is the current state of the Dubai real estate market?
────────────────────────────────────────────────────────────
A: The current state of the Dubai real estate market is one of tempered optimism, transitioning from extraordinary growth to a phase of maturation and consolidation. Transaction volumes are expected to remain high, fueled by population influx and investment interest, even if price growth moderates to single-digit percentages or flattens in some segments. New supply coming online will give end-users more choices and bargaining power, leading to a more balanced and mature phase in the market (2025 Real Estate Market Outlook).

── Sources ───────────────────────────────────────────
  📄 july_market_report_2025.pdf — page 12
  📄 Dubai-Residential-Market-Performance-Q12025.pdf — page 19
  📄 dubai-residential-market-review-special-edition-q3-02025.pdf — page 1
  📄 Dubai-Residential-Market-Performance-Q12025.pdf — page 6



{'answer': 'The current state of the Dubai real estate market is one of tempered optimism, transitioning from extraordinary growth to a phase of maturation and consolidation. Transaction volumes are expected to remain high, fueled by population influx and investment interest, even if price growth moderates to single-digit percentages or flattens in some segments. New supply coming online will give end-users more choices and bargaining power, leading to a more balanced and mature phase in the market (2025 Real Estate Market Outlook).',
 'sources': [{'filename': 'july_market_report_2025.pdf', 'page': 12},
  {'filename': 'Dubai-Residential-Market-Performance-Q12025.pdf', 'page': 19},
  {'filename': 'dubai-residential-market-review-special-edition-q3-02025.pdf',
   'page': 1},
  {'filename': 'Dubai-Residential-Market-Performance-Q12025.pdf', 'page': 6}]}

In [9]:
ask("What are the most popular areas for property investment in Dubai?")

Q: What are the most popular areas for property investment in Dubai?
────────────────────────────────────────────────────────────
A: The most popular areas for property investment in Dubai are Jumeirah Village Circle (JVC), Dubailand, Business Bay, and Dubai Marina for apartments, and Arabian Ranches, Dubai Hills Estate, and Palm Jumeirah for villas. (Market Highlights, page 3)

── Sources ───────────────────────────────────────────
  📄 july_market_report_2025.pdf — page 8
  📄 dubai-residential-market-review-special-edition-q3-02025.pdf — page 19
  📄 july_market_report_2025.pdf — page 2
  📄 july_market_report_2025.pdf — page 6



{'answer': 'The most popular areas for property investment in Dubai are Jumeirah Village Circle (JVC), Dubailand, Business Bay, and Dubai Marina for apartments, and Arabian Ranches, Dubai Hills Estate, and Palm Jumeirah for villas. (Market Highlights, page 3)',
 'sources': [{'filename': 'july_market_report_2025.pdf', 'page': 8},
  {'filename': 'dubai-residential-market-review-special-edition-q3-02025.pdf',
   'page': 19},
  {'filename': 'july_market_report_2025.pdf', 'page': 2},
  {'filename': 'july_market_report_2025.pdf', 'page': 6}]}

In [10]:
ask("What regulations does RERA have for property developers?")

Q: What regulations does RERA have for property developers?
────────────────────────────────────────────────────────────
A: I don't have enough information in the provided documents to answer this.

── Sources ───────────────────────────────────────────
  📄 july_market_report_2025.pdf — page 12
  📄 dubai-residential-market-review-q1-2026.pdf — page 2
  📄 dubai-residential-market-report---q2-2025.pdf — page 4
  📄 july_market_report_2025.pdf — page 4



{'answer': "I don't have enough information in the provided documents to answer this.",
 'sources': [{'filename': 'july_market_report_2025.pdf', 'page': 12},
  {'filename': 'dubai-residential-market-review-q1-2026.pdf', 'page': 2},
  {'filename': 'dubai-residential-market-report---q2-2025.pdf', 'page': 4},
  {'filename': 'july_market_report_2025.pdf', 'page': 4}]}

In [11]:
# ── Test honest fallback ──────────────────────────────────────────────────────
# Should say it doesn't know — not hallucinate
ask("What is the property tax rate in Singapore?")

Q: What is the property tax rate in Singapore?
────────────────────────────────────────────────────────────
A: I don't have enough information in the provided documents to answer this.

── Sources ───────────────────────────────────────────
  📄 Dubai-Residential-Market-Performance-Q12025.pdf — page 18
  📄 dubai-residential-market-review-special-edition-q3-02025.pdf — page 10
  📄 dubai-residential-market-review-special-edition-q3-02025.pdf — page 9



{'answer': "I don't have enough information in the provided documents to answer this.",
 'sources': [{'filename': 'Dubai-Residential-Market-Performance-Q12025.pdf',
   'page': 18},
  {'filename': 'dubai-residential-market-review-special-edition-q3-02025.pdf',
   'page': 10},
  {'filename': 'dubai-residential-market-review-special-edition-q3-02025.pdf',
   'page': 9}]}

In [12]:
# ── Fill this in after running your questions above ───────────────────────────
results = [
    {"question": "Current state of Dubai real estate?",
     "answered": True, "hallucinated": False, "cited_source": True},

    {"question": "Popular investment areas?",
     "answered": True, "hallucinated": False, "cited_source": True},

    {"question": "RERA regulations for developers?",
     "answered": True, "hallucinated": False, "cited_source": True},

    {"question": "Property tax in Singapore?",
     "answered": False, "hallucinated": False, "cited_source": False},
]

answered  = sum(1 for r in results if r["answered"])
no_halluc = sum(1 for r in results if not r["hallucinated"])
cited     = sum(1 for r in results if r["cited_source"])
total     = len(results)

print("── RAG Evaluation ────────────────────────────────────")
print(f"  Questions tested   : {total}")
print(f"  Answered from docs : {answered}/{total}")
print(f"  Hallucination-free : {no_halluc}/{total}")
print(f"  Sources cited      : {cited}/{total}")
print(f"  Grounding rate     : {no_halluc/total*100:.0f}%")

── RAG Evaluation ────────────────────────────────────
  Questions tested   : 4
  Answered from docs : 3/4
  Hallucination-free : 4/4
  Sources cited      : 3/4
  Grounding rate     : 100%
